In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --version | head -n 2
!pip install -q mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-e3b81f7d-e473-9754-fce5-b96690571968, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [21]:
GITHUB_USERNAME = "Beknur0"
print(f"Configured student GitHub identity: {GITHUB_USERNAME}")

Configured student GitHub identity: Beknur0


## Task 1 — Distributed Topology and Hardware-Bound Seed

In [3]:
%%writefile distributed_cuda_solver.py
import hashlib
import subprocess

from mpi4py import MPI
from numba import cuda, float64

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2
GLOBAL_NX = 512
GLOBAL_NY = 512
STEPS = 400
LAMBDA_DAMP = 0.15


def get_gpu_uuid():
    output = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return output.strip().splitlines()[0].strip()


def derive_student_seed(github_user, gpu_uuid):
    text = f"{github_user.strip()}::{gpu_uuid}"
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return int(digest[:16], 16)


def init_distributed_context(github_user):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()

    if size != 4:
        raise RuntimeError(f"Expected 4 MPI ranks, got {size}")

    cuda.select_device(0)

    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = comm.bcast(gpu_uuid, root=0)
    instance_seed = derive_student_seed(github_user, gpu_uuid)

    cart_comm = comm.Create_cart(
        dims=[2, 2],
        periods=[False, False],
        reorder=False,
    )

    cart_rank = cart_comm.Get_rank()
    coords = cart_comm.Get_coords(cart_rank)

    nbr_up, nbr_down = cart_comm.Shift(0, 1)
    nbr_left, nbr_right = cart_comm.Shift(1, 1)

    return {
        "cart_comm": cart_comm,
        "rank": cart_rank,
        "coords": coords,
        "neighbors": (nbr_up, nbr_down, nbr_left, nbr_right),
        "gpu_uuid": gpu_uuid,
        "instance_seed": instance_seed,
    }

Writing distributed_cuda_solver.py


In [4]:
import subprocess

test_code = """
import sys
from distributed_cuda_solver import init_distributed_context

ctx = init_distributed_context(sys.argv[1])
print(
    f"rank={ctx['rank']} coords={ctx['coords']} "
    f"neighbors(up,down,left,right)={ctx['neighbors']} "
    f"seed={ctx['instance_seed']}",
    flush=True,
)
"""

cmd = [
    "mpirun", "--allow-run-as-root", "--oversubscribe",
    "-np", "4",
    "python3", "-c", test_code,
    GITHUB_USERNAME,
]

proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)

if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"Finished with code {proc.returncode}")

rank=0 coords=[0, 0] neighbors(up,down,left,right)=(-2, 2, -2, 1) seed=9768977972034203761
rank=1 coords=[0, 1] neighbors(up,down,left,right)=(-2, 3, 0, -2) seed=9768977972034203761
rank=3 coords=[1, 1] neighbors(up,down,left,right)=(1, -2, 2, -2) seed=9768977972034203761
rank=2 coords=[1, 0] neighbors(up,down,left,right)=(0, -2, -2, 3) seed=9768977972034203761



## Task 2 — Seeded Subdomain Initialization

In [5]:
%%writefile -a distributed_cuda_solver.py

import numpy as np


def initialize_subdomains(ctx):
    ny = GLOBAL_NY // 2
    nx = GLOBAL_NX // 2
    row_rank, col_rank = ctx["coords"]

    dx = 1.0 / (GLOBAL_NX + 1)
    dy = 1.0 / (GLOBAL_NY + 1)

    x = (col_rank * nx + np.arange(nx + 2)) * dx
    y = (row_rank * ny + np.arange(ny + 2)) * dy
    X, Y = np.meshgrid(x, y)

    kappa = 0.16 + 0.08 * np.sin(2.0 * np.pi * X) * np.sin(2.0 * np.pi * Y)

    rng = np.random.default_rng(ctx["instance_seed"])
    u = np.zeros((ny + 2, nx + 2), dtype=np.float64)

    for _ in range(3):
        cx, cy = rng.uniform(0.25, 0.75, size=2)
        amplitude = rng.uniform(0.4, 1.0)
        width = rng.uniform(0.04, 0.08)

        u += amplitude * np.exp(
            -((X - cx) ** 2 + (Y - cy) ** 2) / (2.0 * width ** 2)
        )

    source = 0.001 * np.exp(
        -((X - 0.5) ** 2 + (Y - 0.5) ** 2) / (2.0 * 0.05 ** 2)
    )

    up, down, left, right = ctx["neighbors"]
    if up == MPI.PROC_NULL:
        u[0, :] = 0.0
    if down == MPI.PROC_NULL:
        u[-1, :] = 0.0
    if left == MPI.PROC_NULL:
        u[:, 0] = 0.0
    if right == MPI.PROC_NULL:
        u[:, -1] = 0.0

    return (
        np.ascontiguousarray(u),
        np.ascontiguousarray(kappa),
        np.ascontiguousarray(source),
        dx,
        dy,
    )

Appending to distributed_cuda_solver.py


In [6]:
test_code = """
import sys
import numpy as np
from mpi4py import MPI
from distributed_cuda_solver import init_distributed_context, initialize_subdomains

ctx = init_distributed_context(sys.argv[1])
u, kappa, source, dx, dy = initialize_subdomains(ctx)

assert u.shape == kappa.shape == source.shape == (258, 258)
assert 0.08 - 1e-14 <= kappa.min()
assert kappa.max() <= 0.24 + 1e-14

up, down, left, right = ctx["neighbors"]
if up == MPI.PROC_NULL:
    assert np.all(u[0, :] == 0.0)
if down == MPI.PROC_NULL:
    assert np.all(u[-1, :] == 0.0)
if left == MPI.PROC_NULL:
    assert np.all(u[:, 0] == 0.0)
if right == MPI.PROC_NULL:
    assert np.all(u[:, -1] == 0.0)

print(
    f"rank={ctx['rank']} shape={u.shape} "
    f"kappa=[{kappa.min():.6f}, {kappa.max():.6f}] "
    f"dx={dx:.9f} boundaries=OK",
    flush=True,
)
"""

cmd = [
    "mpirun", "--allow-run-as-root", "--oversubscribe",
    "-np", "4", "python3", "-c", test_code, GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)

if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"Finished with code {proc.returncode}")

rank=3 shape=(258, 258) kappa=[0.159510, 0.239999] dx=0.001949318 boundaries=OK
rank=1 shape=(258, 258) kappa=[0.080001, 0.160490] dx=0.001949318 boundaries=OK
rank=0 shape=(258, 258) kappa=[0.159510, 0.239999] dx=0.001949318 boundaries=OK
rank=2 shape=(258, 258) kappa=[0.080001, 0.160490] dx=0.001949318 boundaries=OK



## Task 3 — GPU Halo Packing and MPI Exchange

In [7]:
%%writefile -a distributed_cuda_solver.py

@cuda.jit
def pack_halos_kernel(u, up, down, left, right, ny, nx):
    t = cuda.grid(1)

    if t < nx:
        up[t] = u[1, t + 1]
        down[t] = u[ny, t + 1]

    if t < ny:
        left[t] = u[t + 1, 1]
        right[t] = u[t + 1, nx]


@cuda.jit
def unpack_halos_kernel(u, up, down, left, right, ny, nx):
    t = cuda.grid(1)

    if t < nx:
        u[0, t + 1] = up[t]
        u[ny + 1, t + 1] = down[t]

    if t < ny:
        u[t + 1, 0] = left[t]
        u[t + 1, nx + 1] = right[t]

Appending to distributed_cuda_solver.py


In [8]:
test_code = """
import sys
import numpy as np
from numba import cuda
from distributed_cuda_solver import (
    init_distributed_context,
    pack_halos_kernel,
    unpack_halos_kernel,
)

ctx = init_distributed_context(sys.argv[1])
ny = nx = 256
threads = 128
blocks = (max(nx, ny) + threads - 1) // threads

host_u = np.arange((ny + 2) * (nx + 2), dtype=np.float64)
host_u = host_u.reshape(ny + 2, nx + 2)

device_u = cuda.to_device(host_u)
buffers = [cuda.device_array(nx, dtype=np.float64) for _ in range(4)]

pack_halos_kernel[blocks, threads](
    device_u, *buffers, ny, nx
)
packed = [buf.copy_to_host() for buf in buffers]

np.testing.assert_array_equal(packed[0], host_u[1, 1:nx + 1])
np.testing.assert_array_equal(packed[1], host_u[ny, 1:nx + 1])
np.testing.assert_array_equal(packed[2], host_u[1:ny + 1, 1])
np.testing.assert_array_equal(packed[3], host_u[1:ny + 1, nx])

recv_host = [
    np.full(nx, 10.0),
    np.full(nx, 20.0),
    np.full(ny, 30.0),
    np.full(ny, 40.0),
]
recv_device = [cuda.to_device(buf) for buf in recv_host]

unpack_halos_kernel[blocks, threads](
    device_u, *recv_device, ny, nx
)
result = device_u.copy_to_host()

np.testing.assert_array_equal(result[0, 1:nx + 1], recv_host[0])
np.testing.assert_array_equal(result[ny + 1, 1:nx + 1], recv_host[1])
np.testing.assert_array_equal(result[1:ny + 1, 0], recv_host[2])
np.testing.assert_array_equal(result[1:ny + 1, nx + 1], recv_host[3])
np.testing.assert_array_equal(result[1:ny + 1, 1:nx + 1],
                              host_u[1:ny + 1, 1:nx + 1])

print(f"rank={ctx['rank']} pack=OK unpack=OK", flush=True)
"""

cmd = [
    "mpirun", "--allow-run-as-root", "--oversubscribe",
    "-np", "4", "python3", "-c", test_code, GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)

if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"Test failed with exit code {proc.returncode}")

rank=0 pack=OK unpack=OK
rank=2 pack=OK unpack=OK
rank=3 pack=OK unpack=OK
rank=1 pack=OK unpack=OK



### Pinned Buffers and Non-Blocking Halo Exchange

In [9]:
%%writefile -a distributed_cuda_solver.py

class HaloBuffers:
    def __init__(self, ny, nx):
        self.ny = ny
        self.nx = nx
        lengths = (nx, nx, ny, ny)

        self.device_send = [
            cuda.device_array(n, dtype=np.float64) for n in lengths
        ]
        self.device_recv = [
            cuda.device_array(n, dtype=np.float64) for n in lengths
        ]

        self.host_send = [
            cuda.pinned_array(n, dtype=np.float64) for n in lengths
        ]
        self.host_recv = [
            cuda.pinned_array(n, dtype=np.float64) for n in lengths
        ]

        for buf in self.host_recv:
            buf.fill(0.0)


def exchange_halos_async(u, ctx, buffers, stream_boundary):
    ny, nx = buffers.ny, buffers.nx
    threads = 128
    blocks = (max(nx, ny) + threads - 1) // threads

    pack_halos_kernel[blocks, threads, stream_boundary](
        u, *buffers.device_send, ny, nx
    )

    for device_buf, host_buf in zip(
        buffers.device_send, buffers.host_send
    ):
        device_buf.copy_to_host(host_buf, stream=stream_boundary)

    stream_boundary.synchronize()

    comm = ctx["cart_comm"]
    neighbors = ctx["neighbors"]

    # Direction order: up, down, left, right.
    send_tags = (0, 1, 2, 3)
    recv_tags = (1, 0, 3, 2)

    requests = []

    for neighbor, tag, host_buf in zip(
        neighbors, recv_tags, buffers.host_recv
    ):
        requests.append(comm.Irecv(host_buf, source=neighbor, tag=tag))

    for neighbor, tag, host_buf in zip(
        neighbors, send_tags, buffers.host_send
    ):
        requests.append(comm.Isend(host_buf, dest=neighbor, tag=tag))

    return requests


def finish_halo_exchange(u, buffers, requests, stream_boundary):
    MPI.Request.Waitall(requests)

    for host_buf, device_buf in zip(
        buffers.host_recv, buffers.device_recv
    ):
        device_buf.copy_to_device(host_buf, stream=stream_boundary)

    ny, nx = buffers.ny, buffers.nx
    threads = 128
    blocks = (max(nx, ny) + threads - 1) // threads

    unpack_halos_kernel[blocks, threads, stream_boundary](
        u, *buffers.device_recv, ny, nx
    )

Appending to distributed_cuda_solver.py


In [10]:
test_code = """
import sys
import numpy as np
from mpi4py import MPI
from numba import cuda
from distributed_cuda_solver import (
    init_distributed_context,
    HaloBuffers,
    exchange_halos_async,
    finish_halo_exchange,
)

ctx = init_distributed_context(sys.argv[1])
ny = nx = 256

host_u = np.zeros((ny + 2, nx + 2), dtype=np.float64)
host_u[1:ny + 1, 1:nx + 1] = ctx["rank"] + 1

device_u = cuda.to_device(host_u)
stream = cuda.stream()
buffers = HaloBuffers(ny, nx)

requests = exchange_halos_async(device_u, ctx, buffers, stream)
finish_halo_exchange(device_u, buffers, requests, stream)
stream.synchronize()

result = device_u.copy_to_host()
up, down, left, right = ctx["neighbors"]

def expected(neighbor):
    return 0.0 if neighbor == MPI.PROC_NULL else float(neighbor + 1)

np.testing.assert_array_equal(result[0, 1:nx + 1], expected(up))
np.testing.assert_array_equal(result[ny + 1, 1:nx + 1], expected(down))
np.testing.assert_array_equal(result[1:ny + 1, 0], expected(left))
np.testing.assert_array_equal(result[1:ny + 1, nx + 1], expected(right))
np.testing.assert_array_equal(
    result[1:ny + 1, 1:nx + 1],
    host_u[1:ny + 1, 1:nx + 1],
)

print(
    f"rank={ctx['rank']} halo_exchange=OK "
    f"received={[expected(n) for n in ctx['neighbors']]}",
    flush=True,
)
"""

cmd = [
    "mpirun", "--allow-run-as-root", "--oversubscribe",
    "-np", "4", "python3", "-c", test_code, GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)

if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"Test failed with exit code {proc.returncode}")

rank=3 halo_exchange=OK received=[2.0, 0.0, 3.0, 0.0]
rank=0 halo_exchange=OK received=[0.0, 3.0, 0.0, 2.0]
rank=2 halo_exchange=OK received=[1.0, 0.0, 0.0, 4.0]
rank=1 halo_exchange=OK received=[0.0, 4.0, 1.0, 0.0]



## Task 4 — Shared-Memory Stencil and Dual-Stream Pipeline

In [12]:
%%writefile -a distributed_cuda_solver.py

@cuda.jit
def tiled_stencil_kernel(
    u, u_next, kappa, source,
    dt, dx, dy, damping,
    r_start, r_end, c_start, c_end,
):
    tile = cuda.shared.array((SHARED_DIM, SHARED_DIM), dtype=float64)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    row = r_start + cuda.blockIdx.y * TILE_DIM + ty
    col = c_start + cuda.blockIdx.x * TILE_DIM + tx

    sy = ty + 1
    sx = tx + 1

    height = u.shape[0]
    width = u.shape[1]

    if row < height and col < width:
        tile[sy, sx] = u[row, col]
    else:
        tile[sy, sx] = 0.0

    if ty == 0:
        if row > 0 and row - 1 < height and col < width:
            tile[0, sx] = u[row - 1, col]
        else:
            tile[0, sx] = 0.0

    if ty == TILE_DIM - 1:
        if row + 1 < height and col < width:
            tile[SHARED_DIM - 1, sx] = u[row + 1, col]
        else:
            tile[SHARED_DIM - 1, sx] = 0.0

    if tx == 0:
        if col > 0 and col - 1 < width and row < height:
            tile[sy, 0] = u[row, col - 1]
        else:
            tile[sy, 0] = 0.0

    if tx == TILE_DIM - 1:
        if col + 1 < width and row < height:
            tile[sy, SHARED_DIM - 1] = u[row, col + 1]
        else:
            tile[sy, SHARED_DIM - 1] = 0.0

    cuda.syncthreads()

    if row < r_end and col < c_end:
        center = tile[sy, sx]
        north = tile[sy - 1, sx]
        south = tile[sy + 1, sx]
        west = tile[sy, sx - 1]
        east = tile[sy, sx + 1]

        kc = kappa[row, col]
        kn = 0.5 * (kc + kappa[row - 1, col])
        ks = 0.5 * (kc + kappa[row + 1, col])
        kw = 0.5 * (kc + kappa[row, col - 1])
        ke = 0.5 * (kc + kappa[row, col + 1])

        diffusion_x = (ke * (east - center) - kw * (center - west)) / (dx * dx)
        diffusion_y = (ks * (south - center) - kn * (center - north)) / (dy * dy)

        u_next[row, col] = center + dt * (
            diffusion_x + diffusion_y
            - damping * center * center * center
            + source[row, col]
        )

Appending to distributed_cuda_solver.py


In [14]:
from pathlib import Path

path = Path("distributed_cuda_solver.py")
text = path.read_text()

replacements = {
    "if row < height and col < width:":
        "if row <= r_end and col <= c_end and row < height and col < width:",

    "if row > 0 and row - 1 < height and col < width:":
        "if row > 0 and row - 1 < height and col <= c_end and col < width:",

    "if row + 1 < height and col < width:":
        "if row + 1 <= r_end and row + 1 < height and col <= c_end and col < width:",

    "if col > 0 and col - 1 < width and row < height:":
        "if col > 0 and col - 1 < width and row <= r_end and row < height:",

    "if col + 1 < width and row < height:":
        "if col + 1 <= c_end and col + 1 < width and row <= r_end and row < height:",
}

for old, new in replacements.items():
    if text.count(old) != 1:
        raise RuntimeError(f"Expected exactly one occurrence: {old}")
    text = text.replace(old, new, 1)

path.write_text(text)
print("Stencil load bounds updated.")

Stencil load bounds updated.


In [13]:
test_code = """
import sys
import numpy as np
from numba import cuda
from distributed_cuda_solver import (
    init_distributed_context,
    initialize_subdomains,
    tiled_stencil_kernel,
)

ctx = init_distributed_context(sys.argv[1])
u, kappa, source, dx, dy = initialize_subdomains(ctx)

dt = 1.0e-6
damping = 0.15
ny = nx = 256

center = u[1:-1, 1:-1]
north = u[:-2, 1:-1]
south = u[2:, 1:-1]
west = u[1:-1, :-2]
east = u[1:-1, 2:]

kc = kappa[1:-1, 1:-1]
kn = 0.5 * (kc + kappa[:-2, 1:-1])
ks = 0.5 * (kc + kappa[2:, 1:-1])
kw = 0.5 * (kc + kappa[1:-1, :-2])
ke = 0.5 * (kc + kappa[1:-1, 2:])

reference = center + dt * (
    (ke * (east - center) - kw * (center - west)) / dx**2
    + (ks * (south - center) - kn * (center - north)) / dy**2
    - damping * center**3
    + source[1:-1, 1:-1]
)

d_u = cuda.to_device(u)
d_kappa = cuda.to_device(kappa)
d_source = cuda.to_device(source)

for r_start, r_end, c_start, c_end in [
    (1, 257, 1, 257),
    (2, 256, 2, 256),
]:
    initial = np.full_like(u, -999.0)
    d_out = cuda.to_device(initial)

    blocks = (
        (c_end - c_start + 15) // 16,
        (r_end - r_start + 15) // 16,
    )

    tiled_stencil_kernel[blocks, (16, 16)](
        d_u, d_out, d_kappa, d_source,
        dt, dx, dy, damping,
        r_start, r_end, c_start, c_end,
    )

    result = d_out.copy_to_host()
    expected = reference[
        r_start - 1:r_end - 1,
        c_start - 1:c_end - 1,
    ]

    np.testing.assert_allclose(
        result[r_start:r_end, c_start:c_end],
        expected,
        rtol=1e-13,
        atol=1e-14,
    )

    mask = np.ones(u.shape, dtype=bool)
    mask[r_start:r_end, c_start:c_end] = False
    assert np.all(result[mask] == -999.0)

print(f"rank={ctx['rank']} stencil=OK", flush=True)
"""

cmd = [
    "mpirun", "--allow-run-as-root", "--oversubscribe",
    "-np", "4", "python3", "-c", test_code, GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)

if proc.returncode != 0:
    print("STDERR:\\n", proc.stderr)
    raise RuntimeError(f"Test failed with exit code {proc.returncode}")

rank=3 stencil=OK
rank=1 stencil=OK
rank=2 stencil=OK
rank=0 stencil=OK



### Serialized and Overlapped Time Steps

In [15]:
%%writefile -a distributed_cuda_solver.py

def launch_stencil(
    u, u_next, kappa, source, dt, dx, dy,
    r_start, r_end, c_start, c_end, stream,
):
    blocks = (
        (c_end - c_start + TILE_DIM - 1) // TILE_DIM,
        (r_end - r_start + TILE_DIM - 1) // TILE_DIM,
    )

    tiled_stencil_kernel[blocks, (TILE_DIM, TILE_DIM), stream](
        u, u_next, kappa, source,
        dt, dx, dy, LAMBDA_DAMP,
        r_start, r_end, c_start, c_end,
    )


def step_overlapped(
    u, u_next, kappa, source, ctx, buffers,
    stream_boundary, stream_interior, dt, dx, dy,
):
    ny, nx = buffers.ny, buffers.nx

    requests = exchange_halos_async(u, ctx, buffers, stream_boundary)

    launch_stencil(
        u, u_next, kappa, source, dt, dx, dy,
        2, ny, 2, nx, stream_interior,
    )

    finish_halo_exchange(u, buffers, requests, stream_boundary)

    launch_stencil(
        u, u_next, kappa, source, dt, dx, dy,
        1, 2, 1, nx + 1, stream_boundary,
    )
    launch_stencil(
        u, u_next, kappa, source, dt, dx, dy,
        ny, ny + 1, 1, nx + 1, stream_boundary,
    )
    launch_stencil(
        u, u_next, kappa, source, dt, dx, dy,
        2, ny, 1, 2, stream_boundary,
    )
    launch_stencil(
        u, u_next, kappa, source, dt, dx, dy,
        2, ny, nx, nx + 1, stream_boundary,
    )

    stream_boundary.synchronize()
    stream_interior.synchronize()


def step_serialized(
    u, u_next, kappa, source, ctx, buffers,
    stream_boundary, dt, dx, dy,
):
    ny, nx = buffers.ny, buffers.nx

    requests = exchange_halos_async(u, ctx, buffers, stream_boundary)
    finish_halo_exchange(u, buffers, requests, stream_boundary)

    launch_stencil(
        u, u_next, kappa, source, dt, dx, dy,
        1, ny + 1, 1, nx + 1, stream_boundary,
    )

    stream_boundary.synchronize()

Appending to distributed_cuda_solver.py


In [16]:
test_code = """
import sys
import numpy as np
from mpi4py import MPI
from numba import cuda
from distributed_cuda_solver import (
    init_distributed_context,
    initialize_subdomains,
    HaloBuffers,
    step_serialized,
    step_overlapped,
)

ctx = init_distributed_context(sys.argv[1])
u, kappa, source, dx, dy = initialize_subdomains(ctx)

dt = 1.0e-6
ny = nx = 256
d_kappa = cuda.to_device(kappa)
d_source = cuda.to_device(source)

def run(overlapped):
    current = cuda.to_device(u)
    next_u = cuda.to_device(np.zeros_like(u))
    buffers = HaloBuffers(ny, nx)
    boundary = cuda.stream()
    interior = cuda.stream()

    for _ in range(5):
        if overlapped:
            step_overlapped(
                current, next_u, d_kappa, d_source, ctx, buffers,
                boundary, interior, dt, dx, dy,
            )
        else:
            step_serialized(
                current, next_u, d_kappa, d_source, ctx, buffers,
                boundary, dt, dx, dy,
            )
        current, next_u = next_u, current

    return current.copy_to_host()

serialized = run(False)
overlapped = run(True)

local_diff = np.max(np.abs(
    serialized[1:-1, 1:-1] - overlapped[1:-1, 1:-1]
))
global_diff = ctx["cart_comm"].allreduce(local_diff, op=MPI.MAX)

assert global_diff < 1e-12
print(
    f"rank={ctx['rank']} five_step_parity=OK "
    f"global_max_diff={global_diff:.3e}",
    flush=True,
)
"""

cmd = [
    "mpirun", "--allow-run-as-root", "--oversubscribe",
    "-np", "4", "python3", "-c", test_code, GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)

if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"Test failed with exit code {proc.returncode}")

rank=2 five_step_parity=OK global_max_diff=0.000e+00
rank=0 five_step_parity=OK global_max_diff=0.000e+00
rank=1 five_step_parity=OK global_max_diff=0.000e+00
rank=3 five_step_parity=OK global_max_diff=0.000e+00



## Task 5 — PTX Audit, Solver, and Benchmark

In [17]:
test_code = """
import sys
import numpy as np
from numba import cuda
from distributed_cuda_solver import (
    init_distributed_context,
    tiled_stencil_kernel,
)

ctx = init_distributed_context(sys.argv[1])

host = np.zeros((258, 258), dtype=np.float64)
d_u = cuda.to_device(host)
d_out = cuda.to_device(host)
d_kappa = cuda.to_device(np.ones_like(host))
d_source = cuda.to_device(host)

tiled_stencil_kernel[(1, 1), (16, 16)](
    d_u, d_out, d_kappa, d_source,
    1.0e-6, 1.0 / 513, 1.0 / 513, 0.15,
    1, 17, 1, 17,
)
cuda.synchronize()

asm = tiled_stencil_kernel.inspect_asm()
if isinstance(asm, dict):
    ptx = "\\n".join(asm.values())
else:
    ptx = asm

counts = {
    "st.shared.f64": ptx.count("st.shared.f64"),
    "ld.shared.f64": ptx.count("ld.shared.f64"),
    "bar.sync": ptx.count("bar.sync"),
}

assert counts["st.shared.f64"] > 0
assert counts["ld.shared.f64"] > 0
assert counts["bar.sync"] > 0

print(f"rank={ctx['rank']} PTX audit=OK counts={counts}", flush=True)
"""

cmd = [
    "mpirun", "--allow-run-as-root", "--oversubscribe",
    "-np", "4", "python3", "-c", test_code, GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)

if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"Test failed with exit code {proc.returncode}")

rank=3 PTX audit=OK counts={'st.shared.f64': 5, 'ld.shared.f64': 5, 'bar.sync': 1}
rank=0 PTX audit=OK counts={'st.shared.f64': 5, 'ld.shared.f64': 5, 'bar.sync': 1}
rank=1 PTX audit=OK counts={'st.shared.f64': 5, 'ld.shared.f64': 5, 'bar.sync': 1}
rank=2 PTX audit=OK counts={'st.shared.f64': 5, 'ld.shared.f64': 5, 'bar.sync': 1}



### Solver Implementation and Manifest Generation

In [18]:
%%writefile -a distributed_cuda_solver.py

import json
import sys
import time


def global_metrics(cart_comm, current, previous, dx, dy):
    current_host = current.copy_to_host()[1:-1, 1:-1]

    local_energy = np.sum(current_host, dtype=np.float64) * dx * dy

    if previous is None:
        local_residual_sq = 0.0
    else:
        previous_host = previous.copy_to_host()[1:-1, 1:-1]
        difference = current_host - previous_host
        local_residual_sq = np.sum(difference * difference, dtype=np.float64)

    local_values = np.array(
        [local_energy, local_residual_sq],
        dtype=np.float64,
    )
    global_values = np.empty_like(local_values)

    cart_comm.Allreduce(local_values, global_values, op=MPI.SUM)

    return float(global_values[0]), float(np.sqrt(global_values[1]))


def inspect_stencil_ptx():
    asm = tiled_stencil_kernel.inspect_asm()
    ptx = "\n".join(asm.values()) if isinstance(asm, dict) else asm

    return {
        "st.shared.f64": ptx.count("st.shared.f64"),
        "ld.shared.f64": ptx.count("ld.shared.f64"),
        "bar.sync": ptx.count("bar.sync"),
    }


def run_mode(mode, ctx, initial_u, kappa, source, dt, dx, dy):
    cart_comm = ctx["cart_comm"]
    rank = ctx["rank"]
    ny = nx = 256

    d_kappa = cuda.to_device(kappa)
    d_source = cuda.to_device(source)
    buffers = HaloBuffers(ny, nx)
    boundary = cuda.stream()
    interior = cuda.stream()

    def execute_step(current, next_u):
        if mode == "overlapped":
            step_overlapped(
                current, next_u, d_kappa, d_source, ctx, buffers,
                boundary, interior, dt, dx, dy,
            )
        else:
            step_serialized(
                current, next_u, d_kappa, d_source, ctx, buffers,
                boundary, dt, dx, dy,
            )

    # Compile and warm up before measuring.
    warm_current = cuda.to_device(initial_u)
    warm_next = cuda.to_device(np.zeros_like(initial_u))
    execute_step(warm_current, warm_next)

    current = cuda.to_device(initial_u)
    next_u = cuda.to_device(np.zeros_like(initial_u))

    energy, residual = global_metrics(cart_comm, current, None, dx, dy)
    trace = [{"step": 0, "energy": energy, "l2_residual": residual}]

    if rank == 0:
        print(
            f"{mode} step=0 energy={energy:.15e} "
            f"l2_residual={residual:.15e}",
            flush=True,
        )

    start_event = cuda.event(timing=True)
    end_event = cuda.event(timing=True)

    cart_comm.Barrier()
    start_time = time.perf_counter()
    start_event.record(boundary)

    for step in range(1, STEPS + 1):
        execute_step(current, next_u)
        current, next_u = next_u, current

        if step % 100 == 0:
            energy, residual = global_metrics(
                cart_comm, current, next_u, dx, dy
            )
            trace.append({
                "step": step,
                "energy": energy,
                "l2_residual": residual,
            })

            if rank == 0:
                print(
                    f"{mode} step={step} energy={energy:.15e} "
                    f"l2_residual={residual:.15e}",
                    flush=True,
                )

    end_event.record(boundary)
    end_event.synchronize()
    local_wall_ms = (time.perf_counter() - start_time) * 1000.0
    local_event_ms = cuda.event_elapsed_time(start_event, end_event)

    wall_ms = cart_comm.allreduce(local_wall_ms, op=MPI.MAX)
    event_ms = cart_comm.allreduce(local_event_ms, op=MPI.MAX)

    return current.copy_to_host(), trace, wall_ms, event_ms


def run_solver(github_user):
    ctx = init_distributed_context(github_user)
    cart_comm = ctx["cart_comm"]
    rank = ctx["rank"]

    initial_u, kappa, source, dx, dy = initialize_subdomains(ctx)

    local_kappa_max = float(np.max(kappa))
    kappa_max = cart_comm.allreduce(local_kappa_max, op=MPI.MAX)
    dt_max = dx * dx / (4.0 * kappa_max)
    dt = 0.5 * dt_max

    if rank == 0:
        print(f"gpu_uuid={ctx['gpu_uuid']}", flush=True)
        print(f"instance_seed={ctx['instance_seed']}", flush=True)
        print(f"mpi_ranks={cart_comm.Get_size()} dims=[2, 2]", flush=True)
        print(
            f"kappa_max={kappa_max:.15e} dx={dx:.15e} "
            f"dt_max={dt_max:.15e} dt={dt:.15e}",
            flush=True,
        )

    serialized, serial_trace, serialized_ms, serialized_event_ms = run_mode(
        "serialized", ctx, initial_u, kappa, source, dt, dx, dy
    )

    overlapped, overlap_trace, overlapped_ms, overlapped_event_ms = run_mode(
        "overlapped", ctx, initial_u, kappa, source, dt, dx, dy
    )

    local_diff = float(np.max(np.abs(
        serialized[1:-1, 1:-1] - overlapped[1:-1, 1:-1]
    )))
    parity_max_diff = cart_comm.allreduce(local_diff, op=MPI.MAX)

    ptx_counts = inspect_stencil_ptx()
    local_ptx_passed = all(count > 0 for count in ptx_counts.values())
    ptx_passed = cart_comm.allreduce(local_ptx_passed, op=MPI.LAND)

    if rank == 0:
        print(f"ptx_counts={ptx_counts}", flush=True)
        print(f"ptx_audit_passed={ptx_passed}", flush=True)
        print(f"parity_max_diff={parity_max_diff:.15e}", flush=True)
        print(
            f"serialized_ms={serialized_ms:.3f} "
            f"overlapped_ms={overlapped_ms:.3f}",
            flush=True,
        )
        print(
            f"serialized_cuda_event_ms={serialized_event_ms:.3f} "
            f"overlapped_cuda_event_ms={overlapped_event_ms:.3f}",
            flush=True,
        )

    if not ptx_passed:
        raise RuntimeError("PTX audit failed")
    if parity_max_diff >= 1e-12:
        raise RuntimeError("Serialized and overlapped results do not match")

    if rank == 0:
        manifest = {
            "github_username": github_user.strip(),
            "gpu_uuid": ctx["gpu_uuid"],
            "instance_seed": ctx["instance_seed"],
            "mpi_ranks": cart_comm.Get_size(),
            "process_grid": [2, 2],
            "steps": STEPS,
            "kappa_max": kappa_max,
            "dx": dx,
            "dy": dy,
            "dt": dt,
            "dt_max": dt_max,
            "serialized_ms": serialized_ms,
            "overlapped_ms": overlapped_ms,
            "serialized_cuda_event_ms": serialized_event_ms,
            "overlapped_cuda_event_ms": overlapped_event_ms,
            "parity_max_diff": parity_max_diff,
            "ptx_counts": ptx_counts,
            "ptx_audit_passed": bool(ptx_passed),
            "serialized_trace": serial_trace,
            "overlapped_trace": overlap_trace,
        }

        with open("submission_manifest.json", "w", encoding="utf-8") as f:
            json.dump(manifest, f, indent=2)

        print("Wrote submission_manifest.json", flush=True)


if __name__ == "__main__":
    if len(sys.argv) != 2:
        raise SystemExit("Usage: python distributed_cuda_solver.py GITHUB_USERNAME")
    run_solver(sys.argv[1])

Appending to distributed_cuda_solver.py


In [19]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py",
    GITHUB_USERNAME,
]

proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)

if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"MPI execution failed with exit code {proc.returncode}")

gpu_uuid=GPU-e3b81f7d-e473-9754-fce5-b96690571968
instance_seed=9768977972034203761
mpi_ranks=4 dims=[2, 2]
kappa_max=2.399992499440621e-01 dx=1.949317738791423e-03 dt_max=3.958178668946418e-06 dt=1.979089334473209e-06
serialized step=0 energy=6.200168542910230e-02 l2_residual=0.000000000000000e+00
serialized step=100 energy=6.200121693692751e-02 l2_residual=6.270046972547441e-03
serialized step=200 energy=6.200075643330039e-02 l2_residual=6.098114570744989e-03
serialized step=300 energy=6.200030295904347e-02 l2_residual=5.935348098649923e-03
serialized step=400 energy=6.199985616372648e-02 l2_residual=5.781057368941544e-03
overlapped step=0 energy=6.200168542910230e-02 l2_residual=0.000000000000000e+00
overlapped step=100 energy=6.200121693692751e-02 l2_residual=6.270046972547441e-03
overlapped step=200 energy=6.200075643330039e-02 l2_residual=6.098114570744989e-03
overlapped step=300 energy=6.200030295904347e-02 l2_residual=5.935348098649923e-03
overlapped step=400 energy=6.199985616

## Analytical and Forensic Responses

### Q1. Memory coalescing and strided column packing

The local array is C-contiguous with 258 float64 elements per row.
Consecutive elements of a halo column are therefore separated by
258 × 8 = 2,064 bytes. A single contiguous DMA transfer cannot copy
only those elements: it would also include the values between them.
When GPU threads read a column, adjacent lanes access widely separated
addresses, so these reads are not coalesced like contiguous row accesses.

One packed column contains 256 × 8 = 2,048 bytes. The smallest contiguous
span covering the same column contains 255 × 258 + 1 = 65,791 float64
elements, or 526,328 bytes. Thus, copying that span transfers 524,280
extra bytes per column. Copying both columns as two separate spans would
transfer 1,048,560 extra bytes per rank per exchange.

The pack kernel performs the strided reads on the GPU and produces
contiguous buffers for the PCIe transfers.

### Q2. Shared-memory indexing and bank conflicts

For an active thread `(tx=0, ty=0)`, the kernel writes its center to
`tile[1, 1]`, and loads the top and left apron values into `tile[0, 1]`
and `tile[1, 0]`. Its stencil reads `tile[1, 1]`, `tile[0, 1]`,
`tile[2, 1]`, `tile[1, 0]`, and `tile[1, 2]`.

For an active thread `(tx=15, ty=15)`, the kernel writes its center to
`tile[16, 16]`, and loads the bottom and right apron values into
`tile[17, 16]` and `tile[16, 17]`. Its stencil reads `tile[16, 16]`,
`tile[15, 16]`, `tile[17, 16]`, `tile[16, 15]`, and `tile[16, 17]`.

Each float64 element occupies two 4-byte banks. However, two banks per
element do not by themselves imply a 2-way conflict: a warp-wide 64-bit
access is serviced in multiple transactions. With a 16 × 16 block, each
warp spans two rows of 16 threads. Contiguous row accesses by each
16-thread group use distinct bank pairs within their transaction.

A hypothetical 16-thread access down a column has a stride of
18 × 8 = 144 bytes, or 36 banks. The starting bank advances by 4 modulo
32, so only eight bank pairs are used by 16 distinct addresses: this
produces a 2-way conflict within that transaction. Actual conflicts
depend on the instruction and active lanes, not merely on the fact that
the data type is float64.

### Q3. PTX audit and synchronization

The PTX audit in my run reported:

- `st.shared.f64`: **5**
- `ld.shared.f64`: **5**
- `bar.sync`: **1**

A 16 × 16 block contains 256 threads, or eight warps. Without
`cuda.syncthreads()`, a warp could read a shared-memory neighbor before
another warp has written it. For example, threads may need values from
an adjacent row loaded by another warp. This is a read-after-write
hazard: the result could contain stale or uninitialized tile values.
The block-wide barrier ensures that the tile loads are complete before
any thread evaluates the stencil.

### Q4. Multiple processes sharing one GPU

My measured wall-clock times for 400 steps were:

- Serialized: **2029.935 ms**
- Overlapped: **3099.968 ms**

The overlapped run was approximately **52.7% slower** in this
measurement. The reported CUDA Event intervals were 2028.835 ms
and 3099.907 ms, respectively. These intervals are not pure kernel
execution times: they also include idle and waiting periods between
operations recorded on the boundary stream. The wall-clock benchmark
also includes the periodic diagnostics.

The four MPI processes create separate CUDA contexts on the same
physical T4. Without CUDA MPS, kernels from different contexts generally
do not execute concurrently on the SMs; the GPU schedules and switches
between contexts. Copy engines and memory bandwidth are shared resources
as well. Separate streams allow overlap within a context when hardware
and dependencies permit it, but they do not guarantee overlap across
independent contexts.

The extra launches, synchronization, host staging, and contention can
outweigh the benefit of overlapping MPI with interior computation.
With four dedicated GPUs across InfiniBand, each rank would have its
own GPU resources, and communication could potentially overlap with
computation more effectively. An improvement is still not guaranteed:
it depends on MPI progress, network latency, message size, and the
available GPU and DMA resources.

### Q5. Blocking sends and MPI.PROC_NULL

MPI implementations commonly use an eager protocol for small messages:
a send may complete after copying data into an internal buffer, even
before the matching receive is posted. Above an implementation-dependent
threshold, a rendezvous protocol may require a handshake with the
receiver before the transfer can complete.

If ranks enter blocking `Send` calls whose matching receives cannot be
posted until those sends return, they can form a circular wait and
deadlock. Small messages may appear to work because of eager buffering,
but MPI does not guarantee that standard `Send` is buffered. The exact
outcome depends on the directional order and matching tags; not every
fixed order necessarily creates the same circular wait.

For Cartesian coordinate `(0, 0)`, the up and left neighbors are
`MPI.PROC_NULL`. Sends to and receives from this special rank complete
without communication. A receive from `MPI.PROC_NULL` does not modify
its receive buffer, so my implementation initializes the corresponding
pinned receive buffers to zero before unpacking them into physical
ghost boundaries.

### Q6. Diffusion stability and energy trace

My run recorded:

- `kappa_max = 0.2399992499440621`
- `dx = 1 / 513 = 0.001949317738791423`
- `dt_max = dx² / (4 × kappa_max) = 3.958178668946418e-6`
- `dt = 1.979089334473209e-6 = 0.5 × dt_max`

Thus, the chosen step is below the stated 2D diffusion CFL limit.
The limit addresses the diffusion component; the nonlinear reaction
and source also need consideration in a general stability analysis.

The Allreduce energy trace was:

| Step | Energy |
|---:|---:|
| 0 | 0.06200168542910230 |
| 100 | 0.06200121693692751 |
| 200 | 0.06200075643330039 |
| 300 | 0.06200030295904347 |
| 400 | 0.06199985616372648 |

The recorded values decrease monotonically for this run. This is
consistent with dissipative behavior, but it is not a general proof
of energy conservation or monotonicity: the system includes a source,
nonlinear damping, and zero Dirichlet boundaries, all of which affect
the sum of `u`. The decreasing trace alone also does not prove numerical
stability for every initial condition or source.